# D4 · Lab 1 — Structured Outputs & Cache Behavior

**Proves:** `output_config.format` guarantees schema-valid JSON; a large cached system prompt shows cache-creation then cache-read; changing the schema/format invalidates the cache.

Requires `ANTHROPIC_API_KEY` set and `pip install anthropic pydantic`. Run the cells top to bottom. Full write-up is in `README.md` (this folder).


## Setup
Imports, the sample invoice, the large cached system prompt, and the JSON schema.


In [ ]:
import os
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY before running (see common/setup-prereq-guide.md)."

from anthropic import Anthropic
from pydantic import BaseModel
import json

client = Anthropic()
invoice_text = "Invoice #12345 from Acme Corp, Due: 2024-01-15, Total: $500.00"
LARGE_SYSTEM_PROMPT = """
You are an expert invoice data extraction assistant used by an enterprise accounts payable team.
Your role is to parse invoice documents and extract structured financial data with high accuracy.

## Core Responsibilities
- Extract all relevant fields from invoices including vendor details, line items, totals, dates, and payment terms.
- Normalize data formats: dates to ISO 8601 (YYYY-MM-DD), amounts to USD decimal strings (e.g. "$1,234.56").
- Identify and flag missing, ambiguous, or potentially fraudulent fields.
- Handle multi-currency invoices by noting the original currency alongside converted USD values.
- Support invoices from international vendors with different tax systems (VAT, GST, HST, etc).

## Field Extraction Rules

### Vendor Information
- vendor_name: Legal business name, not trade name. E.g. "Acme Corporation" not "Acme".
- vendor_address: Full mailing address. Normalize to: Street, City, State/Province, ZIP/Postal, Country.
- vendor_tax_id: EIN, VAT number, or GST number depending on jurisdiction.
- vendor_email: Primary billing contact email.
- vendor_phone: Primary billing contact phone in E.164 format.

### Invoice Metadata
- invoice_number: Unique identifier assigned by vendor. Preserve exact formatting (e.g. "INV-2024-00123").
- invoice_date: Date invoice was issued. ISO 8601.
- due_date: Payment due date. ISO 8601. If net terms given (e.g. Net 30), calculate from invoice_date.
- po_number: Buyer's purchase order number if referenced.
- payment_terms: Raw string as stated (e.g. "Net 30", "2/10 Net 30", "Due on Receipt").

### Financial Fields
- subtotal: Sum of line items before tax and discounts.
- discount_amount: Total discount applied if any.
- tax_amount: Total tax charged (VAT, GST, sales tax, etc).
- tax_rate: Effective tax rate percentage if determinable.
- shipping_amount: Freight or shipping charges if itemized.
- total_amount: Final amount due. Must equal subtotal - discount + tax + shipping.
- amount_paid: Any partial payment already applied.
- amount_due: Remaining balance = total_amount - amount_paid.
- currency: ISO 4217 code (e.g. "USD", "EUR", "GBP", "CAD").

### Line Items (array)
Each line item should contain:
- description: Item or service description.
- quantity: Numeric quantity.
- unit: Unit of measure (e.g. "hours", "each", "kg").
- unit_price: Price per unit.
- line_total: quantity × unit_price.
- item_code: SKU or product code if present.

## Handling Ambiguity

### Missing Fields
- If a field cannot be found in the document, return null for that field.
- Never guess or fabricate values.
- If a field is partially readable (e.g. smudged date), note it as ambiguous in a notes field.

### Date Ambiguity
- If a date is in MM/DD/YYYY or DD/MM/YYYY format and ambiguous, prefer the format consistent with the vendor's country.
- If vendor country is unknown and date is ambiguous (e.g. 01/02/2024), flag it.

### Amount Discrepancies
- If line item totals do not sum to the stated subtotal, note the discrepancy.
- If total does not match subtotal + tax + shipping - discount, flag the inconsistency.

## Quality Checks
Before returning your response, verify:
1. All monetary amounts are consistently in the same currency.
2. Due date is after or equal to invoice date.
3. amount_due = total_amount - amount_paid.
4. Line item totals sum to subtotal.
5. Invoice number is present — if missing, flag as high priority.

## Output Format
Always return valid, minified JSON matching the provided schema exactly.
Do not include markdown, commentary, or explanation outside the JSON structure.
"""

class Invoice(BaseModel):
    vendor: str
    amount: str
    due_date: str

# Build schema and patch it with additionalProperties: false (required by Anthropic API)
invoice_schema = Invoice.model_json_schema()
invoice_schema["additionalProperties"] = False
# DOES invalidate — modifies the schema dict
invoice_schema["properties"]["vendor"]["description"] = "Updated vendor description"


### Run: structured extraction + cache usage
Expect valid JSON and non-zero `cache_creation_input_tokens` on the first run, then `cache_read_input_tokens` on a warm re-run.

**Toggles to try** (edit the cell): uncomment the assistant-prefill turn to see it conflict with JSON outputs; add a `document` block with `citations` enabled to see the 400 error; change a schema property to watch the cache invalidate.


In [ ]:
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    system=[
        {
            "type":"text",
            "text": LARGE_SYSTEM_PROMPT,
            "cache_control": {"type":"ephemeral"}
        }
    ],
    messages=[
        {
            "role": "user",
            "content": f"extract vendor, amount, due_date from this invoice text: {invoice_text}. Respond only with valid JSON."
        }
        #,{
        #    "role": "assistant",  # Prefill: forces Claude to start its response with '{'
        #    "content": "{"
        #}
    ],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": invoice_schema 
        }
    }
    
)

# Reconstruct the full JSON string by prepending the prefill character
#raw_json = "{" + response.content[0].text
raw_json = response.content[0].text
print(json.loads(raw_json))

print("cache_creation_input_tokens: " + str(response.usage.cache_creation_input_tokens))
print("cache_read_input_tokens: " + str(response.usage.cache_read_input_tokens))
